# Cross-model exact validation
Entangled two-qubit stress test with a noncommuting active Fubini–Study metric and active Hessian.

In [ ]:
import numpy as np
from scipy.linalg import expm

I=np.eye(2,dtype=complex); X=np.array([[0,1],[1,0]],complex)
alpha=0.37
V=expm(-1j*alpha*np.kron(X,X))
D=np.diag([0.,1.,3.,5.])
Ham=V@D@V.conj().T
evals=np.linalg.eigvalsh(Ham); E0=evals[0]; Delta=evals[1]-evals[0]; W=evals[-1]-evals[0]
ket00=np.array([1,0,0,0],complex)
u1=np.array([0,1,1,0],complex)/np.sqrt(2)
u2=np.array([0,1,2,0],complex)/np.sqrt(5)
A=1j*(np.outer(u1,ket00)-np.outer(ket00,u1))
B=1j*(np.outer(u2,ket00)-np.outer(ket00,u2))
psi=V@ket00
t1=V@u1; t2=V@u2
T=[t1,t2]
G=np.array([[np.real(np.vdot(T[i],T[j])) for j in range(2)] for i in range(2)])
Hact=np.array([[2*np.real(np.vdot(T[i],(Ham-E0*np.eye(4))@T[j])) for j in range(2)] for i in range(2)])
def invsqrt(M):
    w,Q=np.linalg.eigh(M); return Q@np.diag(1/np.sqrt(w))@Q.T
S0=invsqrt(G)@Hact@invsqrt(G)
comm=np.linalg.norm(G@Hact-Hact@G,2)
gminus,gplus=np.linalg.eigvalsh(G)[[0,-1]]
kappaH=np.linalg.cond(Hact)
epscrit=gminus/2*((Delta/W)*kappaH-1)
print('G=\n',G)
print('H_active=\n',Hact)
print('||[G,H_active]||2=',comm)
print('spec(S0)=',np.linalg.eigvalsh(S0),'kappa(S0)=',np.linalg.cond(S0),'W/Delta=',W/Delta)
assert comm>1e-6
assert np.linalg.cond(S0)<=W/Delta+1e-10

eps=lam=0.02
Xi0=np.array([[0.6,-0.8],[-0.8,-0.6]])
Xi=Xi0/np.linalg.norm(Xi0,2)*eps
M=G+Xi+lam*np.eye(2)
Sl=invsqrt(M)@Hact@invsqrt(M)
K=(W/Delta)*(1+2*eps/gminus)
print('epsilon_crit=',epscrit)
print('kappa(S_lambda)=',np.linalg.cond(Sl),'K=',K,'kappa_H=',kappaH)
assert eps<epscrit and np.linalg.cond(Sl)<K<kappaH
